# Schema Validation Notebook

Runs the two checklist verification steps against all schemas — fetching from GitHub (not local files):

1. **YAML parse check** — equivalent to `python -c "import yaml; yaml.safe_load(open('specs/schema.oold.yaml'))"`
2. **JSON conversion check** — equivalent to `python scripts/convert_to_json.py --check`
3. **Transform smoke-test** — applies `transform.simplified.jsonata` to `docs/example.input.json` and verifies output is non-empty

Schemas are fetched from their tagged raw GitHub URLs (from `x-schema-uri` in each schema).

Run all cells top to bottom. A summary table at the end shows pass/fail per schema.

In [ ]:
import json, yaml, requests, io
from pathlib import Path
from typing import Optional

REPO = "https://raw.githubusercontent.com/semantic-dataspace/semantic-schemas"
BRANCH = "kiran"  # fallback branch for schemas without a tag

# ── Helpers ──────────────────────────────────────────────────────────────────

def fetch(url: str) -> Optional[str]:
    """Fetch text from URL; return None on 404."""
    r = requests.get(url, timeout=15)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.text


def raw_url(ref: str, path: str) -> str:
    """Build a raw GitHub URL for a given tag/branch and repo-relative path."""
    return f"{REPO}/{ref}/{path}"


def schema_ref_from_yaml(yaml_text: str) -> str:
    """Extract the tag/branch embedded in x-schema-uri."""
    d = yaml.safe_load(yaml_text)
    uri = d.get("x-schema-uri", "")
    if not uri:
        return BRANCH
    # URI pattern: .../semantic-schemas/<tag>/schemas/...
    return uri.split("semantic-schemas/")[-1].split("/schemas/")[0]


print("Setup done.")

---
## Schema list

All schemas are read from `schemas/manifest.json` on the kiran branch.
Each schema's tag is extracted from `x-schema-uri` inside the YAML, so the
correct versioned raw URL is used for every schema.

In [ ]:
manifest_url = raw_url(BRANCH, "schemas/manifest.json")
manifest = json.loads(fetch(manifest_url))

schemas = manifest["tree"]
print(f"Loaded {len(schemas)} schemas from manifest")
for s in schemas:
    print(f"  {s['path'].replace('schemas/', '').replace('/specs/schema.oold.yaml', ''):<55} v{s['version']}")

---
## Run checks

For each schema:
1. Fetch `specs/schema.oold.yaml` from the tagged URL
2. Parse with `yaml.safe_load` — catches YAML syntax errors
3. Convert to JSON with `json.dumps(yaml.safe_load(...))` — catches non-serialisable objects
4. Fetch `docs/example.input.json` and apply `specs/transform.simplified.jsonata` — catches broken transforms

In [ ]:
try:
    from jsonata import Jsonata
    HAS_JSONATA = True
except ImportError:
    HAS_JSONATA = False
    print("⚠  jsonata not installed — transform check will be skipped")
    print("   pip install jsonata")

results = []

for entry in schemas:
    schema_path = entry["path"]          # e.g. schemas/characterization/generic/PMDCo/specs/schema.oold.yaml
    schema_dir  = schema_path.rsplit("/specs/", 1)[0]  # e.g. schemas/characterization/generic/PMDCo
    label       = schema_dir.replace("schemas/", "")

    row = {"schema": label, "version": entry["version"],
           "yaml_parse": None, "json_convert": None, "transform": None, "error": ""}

    # ── Step 1: fetch YAML from branch (to read tag), then refetch from tag ──
    try:
        yaml_branch_url = raw_url(BRANCH, schema_path)
        yaml_text = fetch(yaml_branch_url)
        if yaml_text is None:
            row["error"] = "schema.oold.yaml not found on branch"
            results.append(row)
            continue
        tag = schema_ref_from_yaml(yaml_text)
        if tag != BRANCH:
            tagged_url = raw_url(tag, schema_path)
            tagged_text = fetch(tagged_url)
            yaml_text = tagged_text if tagged_text else yaml_text
    except Exception as e:
        row["error"] = f"fetch error: {e}"
        results.append(row)
        continue

    # ── Step 2: YAML parse ────────────────────────────────────────────────────
    try:
        parsed = yaml.safe_load(yaml_text)
        row["yaml_parse"] = "✓"
    except yaml.YAMLError as e:
        row["yaml_parse"] = "✗"
        row["error"] = f"YAML: {e}"
        results.append(row)
        continue

    # ── Step 3: JSON conversion ───────────────────────────────────────────────
    try:
        json_str = json.dumps(parsed, ensure_ascii=False)
        json.loads(json_str)   # round-trip check
        row["json_convert"] = "✓"
    except (TypeError, ValueError) as e:
        row["json_convert"] = "✗"
        row["error"] = f"JSON: {e}"
        results.append(row)
        continue

    # ── Step 4: transform smoke-test ─────────────────────────────────────────
    example_path   = f"{schema_dir}/docs/example.input.json"
    transform_path = f"{schema_dir}/specs/transform.simplified.jsonata"

    example_url   = raw_url(BRANCH, example_path)
    transform_url = raw_url(BRANCH, transform_path)

    example_text   = fetch(example_url)
    transform_text = fetch(transform_url)

    if not example_text or not transform_text:
        row["transform"] = "—"   # no example or no transform (checklist §8 exception)
    elif not HAS_JSONATA:
        row["transform"] = "skip"
    else:
        try:
            data   = json.loads(example_text)
            output = Jsonata(transform_text).evaluate(data)
            row["transform"] = "✓" if output else "✗ (empty output)"
        except Exception as e:
            row["transform"] = "✗"
            row["error"]    += f" | transform: {e}"

    results.append(row)
    print(f"  {label:<55} yaml={row['yaml_parse']} json={row['json_convert']} transform={row['transform']}")

print("\nDone.")

---
## Summary

In [ ]:
pass_all  = [r for r in results if r["yaml_parse"]=="✓" and r["json_convert"]=="✓" and r["transform"] in ("✓","—","skip")]
fail_yaml = [r for r in results if r["yaml_parse"] == "✗"]
fail_json = [r for r in results if r["yaml_parse"] == "✓" and r["json_convert"] == "✗"]
fail_xfm  = [r for r in results if "✗" in str(r["transform"])]

print(f"Total schemas : {len(results)}")
print(f"All pass      : {len(pass_all)}")
print(f"YAML failures : {len(fail_yaml)}")
print(f"JSON failures : {len(fail_json)}")
print(f"Transform fail: {len(fail_xfm)}")
print()

col_w = 55
print(f"{'Schema':<{col_w}} {'Ver':<8} {'YAML':<6} {'JSON':<6} {'XFM':<6} Error")
print("-" * 110)
for r in sorted(results, key=lambda x: (x["yaml_parse"]!="✓", x["json_convert"]!="✓", x["schema"])):
    err = r["error"][:50] if r["error"] else ""
    print(f"{r['schema']:<{col_w}} {r['version']:<8} {r['yaml_parse'] or '?':<6} {r['json_convert'] or '?':<6} {r['transform'] or '?':<6} {err}")

if fail_yaml or fail_json or fail_xfm:
    print()
    print("✗ Some checks failed — fix before merging to main.")
else:
    print()
    print("✓ All schemas pass YAML parse and JSON conversion.")